# Analysis Checks and Tableau Export

This notebook has two jobs. First, it checks the summary tables that `04_analysis.sql` builds, so you can confirm the numbers look right before they go anywhere. Second, it saves each of those tables as a CSV file so Tableau Public can read them.

The notebook does not change the data. Every number comes from the `agg_` tables that the SQL scripts already built.

The checks cover four things. The first confirms that every summary table exists and has rows. The second prints the headline figures, such as the busiest hour and the card payment share, so they can be compared with the dashboard. The third checks that the row counts line up from the raw table to the cleaned table to the zone table. The fourth exports the tables to CSV.

## 0. Config

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

DB_USER     = ''   # Put your database username
DB_PASSWORD = ''   # Put your database password
DB_HOST     = ''   # Put your database host
DB_PORT     = ''   # Put your database port
DB_NAME     = ''   # Put the name of your database

url = URL.create(
    drivername='postgresql+psycopg2',
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)
engine = create_engine(url)

# Export destination
EXPORT_DIR = r'' #Put the export file path
os.makedirs(EXPORT_DIR, exist_ok=True)

print('Connected.')
print(f'Exporting to: {EXPORT_DIR}')

## 1. Row-count check

Confirms every aggregation table exists and has rows.  
A zero-row table means the upstream SQL did not run or filtered everything out.

In [ ]:
AGG_TABLES = [
    'agg_hourly_demand',
    'agg_dow_demand',
    'agg_monthly_demand',
    'agg_hour_dow_heatmap',
    'agg_pu_borough_demand',
    'agg_do_borough_demand',
    'agg_top_pu_zones',
    'agg_top_do_zones',
    'agg_borough_flow',
    'agg_payment_type',
    'agg_tip_analysis',
    'agg_fare_components',
    'agg_rate_code',
    'agg_speed_by_borough',
    'agg_passenger_dist',
    'agg_airport_trips',
    'agg_suspicious_flags',
]

rows = []
with engine.connect() as conn:
    for t in AGG_TABLES:
        result = conn.execute(text(f'SELECT COUNT(*) FROM {t}'))
        rows.append({'table': t, 'row_count': result.scalar()})

check_df = pd.DataFrame(rows)
check_df['status'] = check_df['row_count'].apply(lambda n: 'OK' if n > 0 else 'EMPTY')
print(check_df.to_string(index=False))

empty = check_df[check_df['status'] == 'EMPTY']
if empty.empty:
    print('\nAll tables have data.')
else:
    print(f'\nWARNING: {len(empty)} empty table(s): {list(empty["table"])}')